# 01 · EDA y calidad de datos — SportSmart Perú

Fase CRISP-DM: **Comprensión de los datos** (Presentación 1, semana 7).

Este notebook ejecuta `src/data/eda_calidad.py` y presenta sus resultados en tablas y gráficos.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().resolve().parents[1]  # notebooks/01_eda -> raíz
sys.path.insert(0, str(ROOT))
from src.config import RAW, REPORTS, FIGURES
import pandas as pd
import matplotlib.pyplot as plt

## 1. Ejecutar el diagnóstico (genera `reports/eda.json`)

In [ ]:
import runpy, contextlib, io
with contextlib.redirect_stdout(io.StringIO()):
    runpy.run_path(str(ROOT / 'src' / 'data' / 'eda_calidad.py'))
res = json.loads((REPORTS / 'eda.json').read_text(encoding='utf-8'))
print('Total de registros:', res['total_registros'])

## 2. Inventario de tablas

In [ ]:
pd.DataFrame(res['inventario']).T.sort_values('filas', ascending=False)

## 3. Calidad de datos

In [ ]:
cal = pd.DataFrame({k: {'% celdas nulas': v['pct_celdas_nulas'], 'duplicados exactos': v['duplicados_exactos']} for k, v in res['calidad'].items()}).T
cal

In [ ]:
{k: {kk: vv for kk, vv in v.items() if kk not in ('nulos', 'pct_celdas_nulas', 'duplicados_exactos')} for k, v in res['calidad'].items()}

## 4. Ventas en el tiempo

In [ ]:
e = res['eda']
vm = pd.Series(e['ventas_mensuales'])
ax = vm.plot(figsize=(11, 4), title='Ventas mensuales (S/)')
ax.set_xlabel('Mes'); ax.set_ylabel('S/')
FIGURES.mkdir(parents=True, exist_ok=True)
plt.tight_layout(); plt.savefig(FIGURES / 'ventas_mensuales.png', dpi=150); plt.show()

In [ ]:
pd.Series(e['indice_estacional_mes']).plot.bar(figsize=(8, 3), title='Índice estacional por mes'); plt.axhline(1, ls='--', c='gray'); plt.show()

## 5. Clientes y productos

In [ ]:
pd.Series(e['ventas_categoria_pct']).plot.barh(title='% de ventas por categoría'); plt.show()
print('Pedidos por cliente:', e['pedidos_por_cliente'])
print('% clientes sin compra en 180 días:', e['pct_clientes_sin_compra_180d'])
print('% ventas del top 20% de clientes:', e['pct_ventas_top20_clientes'])

In [ ]:
pd.DataFrame(e['top_pares'])  # vista previa para 03_association

## 6. Hallazgos para el negocio

- _Completar con las conclusiones del equipo (abandono, concentración de ventas, efecto de los retrasos en la satisfacción, estacionalidad)._